# 1. A standard FTH experiment at normal incidence

Follow sample design → local interaction and multislice → exit field → physical detector → acquisition. Figure 1 sketches the experiment and code flow; Figure 2 displays the simulated stages. CR and CL use identical geometry and one shared synthetic photon scale. [API guide](../API_GUIDE.md).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src/scattering_calculator').is_dir())
PAPER = ROOT / 'paper/scattering_calculator'
sys.path.insert(0, str(PAPER))
import experiments as ex
import workflow as wf

def show(name):
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(plt.imread(OUT / (name + '.png')))
    ax.axis('off')
    plt.show()


## Editable configuration
The object opening removes only Au; the reference opening penetrates every layer. Change the sample builder for other masks or magnetization inputs.

In [ ]:
# EDIT HERE: sample dimensions are nm; detector lengths are m.
sample = ex.FTHConfig(
    energy_eV=778., n=192, dx_nm=10.,
    layers_nm=(('Au',300.), ('SiN',20.), ('Pt',2.), ('Co',10.), ('Pt',2.)),
    max_slice_nm=5., object_radius_nm=180., reference_radius_nm=30.,
    reference_xy_nm=(600.,0.), beam_sigma_nm=650.,
    domain_period_nm=110., propagate=True,
    energies_eV=(772.,775.,778.,780.,783.,790.,795.,800.),
)
camera = wf.DetectorGeometry(shape=(128,128), pixel_size_m=13.5e-6,
                             distance_m=.05, method='fraunhofer')
effects = ex.DetectorEffectsConfig()
OUT = PAPER / 'results' / 'notebook_fth_basics'


In [ ]:
case, ideal, acquisition = wf.baseline(OUT, sample, camera, effects)
show('fig01_setup_workflow')
show('fig02_fth_chain')


## Inspect the API outputs
Complex exit fields retain phase; intensity alone cannot be propagated coherently. `case["images"]` is on the common FFT q grid. `ideal` and acquisition arrays are on the physical camera. Counts use a synthetic budget, not absolute flux calibration.

In [ ]:
print('Exit fields:', case['exits'].shape)
print('FFT intensities:', case['images'].shape)
print('Physical detector:', ideal.shape)
print('Shared budget scale:', acquisition['scale'])
print('Saved to:', OUT)
